### Function to load sample questions from dataset

In [24]:
!pip install python-dotenv datasets


[notice] A new release of pip is available: 25.2 -> 26.2
[notice] To update, run: pip install --upgrade pip


In [25]:
import numpy as np

In [26]:
import os
from dotenv import load_dotenv

# Load the .env file
load_dotenv()

# Access them directly from os.environ
print("HF_TOKEN set:", "HF_TOKEN" in os.environ)
openrouter_token = os.environ['OPENROUTER_TOKEN']

HF_TOKEN set: True


In [27]:
import sys, os


def _add_ragbench_lib_to_path():
    for candidate in (os.getcwd(), os.path.join(os.getcwd(), "delucion_dataset")):
        if os.path.isdir(os.path.join(candidate, "ragbench_lib")) and candidate not in sys.path:
            sys.path.insert(0, candidate)
            return


_add_ragbench_lib_to_path()

from ragbench_lib.data_loading import load_rag_bench_data, RAGBENCH_CONFIGS


### call the function to load sample

In [ ]:
import pandas as pd

# Load 50 samples to have enough for 40 unique test samples
DATASET_NAME = "delucionqa"  # feeds both load_rag_bench_data() and vector store naming below
docs_df = load_rag_bench_data(DATASET_NAME, split="test", num_samples=50)
print(f"Total documents: {len(docs_df)}")

## Chunking Strategy Optimization (Based on RAG Paper)

Testing 5 different chunking methods to find optimal performance

In [29]:
import tiktoken

# Initialize token counter
encoding = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    """Count tokens in text"""
    return len(encoding.encode(text))

# Chunking Strategy 1: Current Baseline (250 chars)
def chunking_method_1_baseline(docs_df):
    """Method 1: Current - 250 char chunks"""
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=250,
        chunk_overlap=50,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    
    chunks = []
    for _, doc in docs_df.iterrows():
        splits = splitter.split_text(doc["text"])
        for chunk_idx, chunk_text in enumerate(splits):
            chunks.append({
                "method": "M1_Baseline_250chars",
                "chunk_id": f"{doc['doc_id']}_c{chunk_idx}",
                "text": chunk_text,
                "tokens": count_tokens(chunk_text),
                "chars": len(chunk_text),
                "row_id": doc['row_id'],
                "doc_id": doc['doc_id'],
            })
    
    return chunks

# Chunking Strategy 2: Optimized Size (512 tokens - paper optimal)
def chunking_method_2_optimized_size(docs_df):
    """Method 2: Optimized - 512 token chunks (paper recommendation)"""
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    
    # 512 tokens ≈ 2048 chars, overlap ≈ 50 tokens
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=2048,
        chunk_overlap=200,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    
    chunks = []
    for _, doc in docs_df.iterrows():
        splits = splitter.split_text(doc["text"])
        for chunk_idx, chunk_text in enumerate(splits):
            chunks.append({
                "method": "M2_Optimized_512tokens",
                "chunk_id": f"{doc['doc_id']}_c{chunk_idx}",
                "text": chunk_text,
                "tokens": count_tokens(chunk_text),
                "chars": len(chunk_text),
                "row_id": doc['row_id'],
                "doc_id": doc['doc_id'],
            })
    
    return chunks

# Chunking Strategy 3: Sliding Window (paper winner)
def chunking_method_3_sliding_window(docs_df):
    """Method 3: Sliding Window - 512 token chunks with 50% overlap"""
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    
    # 512 tokens, 50% overlap = 256 tokens
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=2048,
        chunk_overlap=1024,  # 50% overlap for sliding window
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    
    chunks = []
    for _, doc in docs_df.iterrows():
        splits = splitter.split_text(doc["text"])
        for chunk_idx, chunk_text in enumerate(splits):
            chunks.append({
                "method": "M3_SlidingWindow_512tokens",
                "chunk_id": f"{doc['doc_id']}_c{chunk_idx}",
                "text": chunk_text,
                "tokens": count_tokens(chunk_text),
                "chars": len(chunk_text),
                "row_id": doc['row_id'],
                "doc_id": doc['doc_id'],
            })
    
    return chunks

# Chunking Strategy 4: Small-to-Big (paper alternative)
def chunking_method_4_small_to_big(docs_df):
    """Method 4: Small-to-Big - Small chunks for matching, large for context"""
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    
    small_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1024,    # 256 tokens
        chunk_overlap=200,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    
    chunks = []
    for _, doc in docs_df.iterrows():
        splits = small_splitter.split_text(doc["text"])
        for chunk_idx, chunk_text in enumerate(splits):
            chunks.append({
                "method": "M4_SmallToBig",
                "chunk_id": f"{doc['doc_id']}_c{chunk_idx}",
                "text": chunk_text,
                "tokens": count_tokens(chunk_text),
                "chars": len(chunk_text),
                "row_id": doc['row_id'],
                "doc_id": doc['doc_id'],
            })
    
    return chunks

# Chunking Strategy 5: Optimized with Metadata
def chunking_method_5_with_metadata(docs_df):
    """Method 5: Optimized + Metadata - Sliding window with enriched info"""
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=2048,
        chunk_overlap=1024,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    
    chunks = []
    for _, doc in docs_df.iterrows():
        splits = splitter.split_text(doc["text"])
        for chunk_idx, chunk_text in enumerate(splits):
            # Extract first line as potential section header
            first_line = chunk_text.split('\n')[0][:100]
            
            chunks.append({
                "method": "M5_SlidingWindow_WithMetadata",
                "chunk_id": f"{doc['doc_id']}_c{chunk_idx}",
                "text": chunk_text,
                "tokens": count_tokens(chunk_text),
                "chars": len(chunk_text),
                "row_id": doc['row_id'],
                "doc_id": doc['doc_id'],
                "metadata": first_line,
            })
    
    return chunks

print("✓ Defined 5 chunking strategies")

✓ Defined 5 chunking strategies


In [30]:
def compare_chunking_methods(docs_df):
    """Compare all 5 chunking methods"""
    
    print("\n" + "="*100)
    print("CHUNKING STRATEGY COMPARISON (Based on RAG Paper)")
    print("="*100 + "\n")
    
    methods = [
        ("M1 (Baseline)", chunking_method_1_baseline),
        ("M2 (Optimized Size)", chunking_method_2_optimized_size),
        ("M3 (Sliding Window)", chunking_method_3_sliding_window),
        ("M4 (Small-to-Big)", chunking_method_4_small_to_big),
        ("M5 (With Metadata)", chunking_method_5_with_metadata),
    ]
    
    comparison_data = []
    
    for method_name, method_func in methods:
        print(f"Processing {method_name}...", end=" ")
        chunks = method_func(docs_df)
        
        token_sizes = [c["tokens"] for c in chunks]
        char_sizes = [c["chars"] for c in chunks]
        
        stats = {
            "Method": method_name,
            "Total Chunks": len(chunks),
            "Avg Tokens": f"{np.mean(token_sizes):.0f}",
            "Min Tokens": f"{min(token_sizes)}",
            "Max Tokens": f"{max(token_sizes)}",
            "Avg Chars": f"{np.mean(char_sizes):.0f}",
            "Std Dev": f"{np.std(token_sizes):.0f}",
        }
        
        comparison_data.append(stats)
        print(f"✓ ({len(chunks)} chunks)")
    
    df_comparison = pd.DataFrame(comparison_data)
    print("\n" + "="*100)
    print("CHUNKING STATISTICS")
    print("="*100)
    display(df_comparison)
    
    print("\n" + "-"*100)
    print("Paper Recommendations:")
    print("  • Optimal chunk size: 512 tokens (M2, M3, M5)")
    print("  • Best technique: Sliding window (M3)")
    print("  • Expected improvement: +7-13% in Context Relevance")
    print("-"*100)
    
    return comparison_data

# Run comparison
chunking_comparison = compare_chunking_methods(docs_df)


CHUNKING STRATEGY COMPARISON (Based on RAG Paper)

Processing M1 (Baseline)... ✓ (1118 chunks)
Processing M2 (Optimized Size)... ✓ (186 chunks)
Processing M3 (Sliding Window)... ✓ (194 chunks)
Processing M4 (Small-to-Big)... ✓ (285 chunks)
Processing M5 (With Metadata)... ✓ (194 chunks)

CHUNKING STATISTICS


,Method,Total Chunks,Avg Tokens,Min Tokens,Max Tokens,Avg Chars,Std Dev
0,M1 (Baseline),1118,38,5,81,183,11
1,M2 (Optimized Size),186,231,5,480,1099,127
2,M3 (Sliding Window),194,260,5,480,1241,124
3,M4 (Small-to-Big),285,161,5,244,766,55
4,M5 (With Metadata),194,260,5,480,1241,124



----------------------------------------------------------------------------------------------------
Paper Recommendations:
  • Optimal chunk size: 512 tokens (M2, M3, M5)
  • Best technique: Sliding window (M3)
  • Expected improvement: +7-13% in Context Relevance
----------------------------------------------------------------------------------------------------


## Next Step: Test Each Chunking Method

Instructions:
1. Choose a chunking method (M1-M5)
2. Run the cell below to rebuild vector store with selected method
3. Run the evaluation loop to measure TRACe metrics
4. Compare results across methods

In [ ]:
def rebuild_vector_store_with_method(selected_method_name: str, docs_df, embedding_model):
    """
    Rebuild vector store using selected chunking method
    
    Parameters:
    - selected_method_name: 'M1', 'M2', 'M3', 'M4', or 'M5'
    - docs_df: input documents
    - embedding_model: embedding model to use
    
    Returns:
    - vector_store: new Chroma vector store
    - documents: Document objects
    """
    from langchain_core.documents import Document
    from langchain_chroma import Chroma
    from ragbench_lib.vector_store import get_persist_dir, vector_store_names
    
    # Method mapping
    methods = {
        'M1': chunking_method_1_baseline,
        'M2': chunking_method_2_optimized_size,
        'M3': chunking_method_3_sliding_window,
        'M4': chunking_method_4_small_to_big,
        'M5': chunking_method_5_with_metadata,
    }
    
    if selected_method_name not in methods:
        print(f"Invalid method. Choose from: {list(methods.keys())}")
        return None, None
    
    print(f"\n{'='*80}")
    print(f"REBUILDING VECTOR STORE WITH: {selected_method_name}")
    print(f"{'='*80}\n")
    
    # Get chunks using selected method
    print(f"Applying {selected_method_name} chunking strategy...", end=" ")
    chunks = methods[selected_method_name](docs_df)
    print(f"✓ Created {len(chunks)} chunks")
    
    # Convert to Document objects
    documents = [
        Document(
            page_content=chunk["text"],
            metadata={
                "chunk_id": chunk["chunk_id"],
                "method": chunk["method"],
                "tokens": chunk["tokens"],
                "row_id": chunk["row_id"],
            }
        )
        for chunk in chunks
    ]
    
    # Build new vector store (fresh directory per call -- Chroma caches its
    # client per persist path in-process, so deleting and reusing the same
    # path here would hand the next write a stale cached client)
    print(f"Building new vector store with {len(documents)} documents...", end=" ")
    prefix, collection_name = vector_store_names(DATASET_NAME, selected_method_name)
    CHROMA_PATH = get_persist_dir(prefix)
    vector_store = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        collection_name=collection_name,
        persist_directory=CHROMA_PATH,
    )
    print("✓")
    
    return vector_store, documents


# Example: Uncomment to test Method 3 (Sliding Window - Paper Winner)
# print("\nNote: Method 3 (Sliding Window) is the paper's recommended approach")
# print("Uncomment below to rebuild vector store and run evaluation with M3\n")

# vector_store_m3, docs_m3 = rebuild_vector_store_with_method('M3', docs_df, embedding_model)
# print(f"\nVector store rebuilt. Ready for TRACe metrics evaluation.")

In [32]:
def run_quick_experiment(method_name, docs_df, embedding_model, llm, prompt, num_samples=10):
    """
    Quick experiment: rebuild vector store, create RAG chain, and evaluate
    Returns: rag_chain, retriever, results dictionary
    """
    print(f"\n{'='*80}\nQUICK EXPERIMENT: {method_name} ({num_samples} samples)\n{'='*80}\n")
    
    # Step 1: Rebuild vector store
    vector_store_exp, _ = rebuild_vector_store_with_method(method_name, docs_df, embedding_model)
    
    # Step 2: Create retriever
    retriever_exp = vector_store_exp.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 8}
    )
    
    # Step 3: Create RAG chain
    rag_chain_exp = ({
        "context": retriever_exp | format_docs,
        "question": RunnablePassthrough(),
    } | prompt | llm | StrOutputParser())
    
    # Step 4: Run evaluation on sample
    unique_samples = docs_df.drop_duplicates(subset=['row_id']).head(num_samples).reset_index(drop=True)
    
    metrics_list = []
    for i, row in unique_samples.iterrows():
        question = row['question']
        try:
            my_response = rag_chain_exp.invoke(question)
            retrieved_docs = retriever_exp.invoke(question)
            retrieved_texts = [doc.page_content for doc in retrieved_docs]
            
            annotation = annotate_response_for_metrics(retrieved_texts, question, my_response)
            
            if annotation['success']:
                metrics_list.append({
                    'context_relevance': compute_context_relevance_from_annotation(retrieved_texts, annotation),
                    'utilization': compute_utilization_from_annotation(retrieved_texts, annotation),
                    'completeness': compute_completeness_from_annotation(annotation),
                    'adherence': compute_adherence_from_annotation(annotation),
                })
                print(f"[{i+1}/{num_samples}] ✓")
            else:
                print(f"[{i+1}/{num_samples}] ✗ Annotation failed")
        except Exception as e:
            print(f"[{i+1}/{num_samples}] ✗ Error")
    
    # Step 5: Aggregate results
    if metrics_list:
        avg_metrics = {
            'context_relevance': np.mean([m['context_relevance'] for m in metrics_list]),
            'utilization': np.mean([m['utilization'] for m in metrics_list]),
            'completeness': np.mean([m['completeness'] for m in metrics_list]),
            'adherence': np.mean([m['adherence'] for m in metrics_list]),
        }
        
        print(f"\n{'='*80}")
        print(f"RESULTS: {method_name}")
        print(f"{'='*80}")
        print(f"  Context Relevance: {avg_metrics['context_relevance']:.4f}")
        print(f"  Utilization:       {avg_metrics['utilization']:.4f}")
        print(f"  Completeness:      {avg_metrics['completeness']:.4f}")
        print(f"  Adherence:         {avg_metrics['adherence']:.4f}")
        print(f"{'='*80}\n")
        
        return rag_chain_exp, retriever_exp, avg_metrics
    else:
        print("No successful evaluations!")
        return rag_chain_exp, retriever_exp, None

print("✓ Quick experiment function defined")

✓ Quick experiment function defined


### installing required libraries

In [33]:
!pip install -qU \
    "opentelemetry-api>=1.36.0,<1.39.0" \
    "opentelemetry-sdk>=1.36.0,<1.39.0" \
    langchain-core \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-chroma \
    langchain-openai


[notice] A new release of pip is available: 25.2 -> 26.2
[notice] To update, run: pip install --upgrade pip


### Phase 1 : Splitting the documents into Chunks

In [34]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,           # characters (not tokens)
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],  # tries each in order
)


chunks = []
for _, doc in docs_df.iterrows():
    splits = splitter.split_text(doc["text"])
    for chunk_idx, chunk_text in enumerate(splits):
        chunks.append({
            "chunk_id":     f"{doc['doc_id']}_c{chunk_idx}",
            "doc_id":       doc["doc_id"],
            "row_id":       doc["row_id"],
            "doc_pos":      doc["doc_pos"],
            "chunk_idx":    chunk_idx,
            "total_chunks": len(splits),
            "text":         chunk_text,
        })


### Converting them to Langchain Documents

In [35]:
from langchain_core.documents import Document
from langchain_chroma import Chroma

# Convert dicts → Document objects
# 'text' becomes page_content, everything else goes into metadata
documents = [
    Document(
        page_content=chunk["text"],
        metadata={
            "chunk_id"    : chunk["chunk_id"],
            "doc_id"      : chunk["doc_id"],
            "row_id"      : chunk["row_id"],
            "doc_pos"     : chunk["doc_pos"],
        }
    )
    for chunk in chunks
]

print(f"Converted {len(documents)} dicts → Document objects")
print(f"Sample page_content : {documents[0].page_content[:100]}")
print(f"Sample metadata     : {documents[0].metadata}")

Converted 1118 dicts → Document objects
Sample page_content : Closing To close the tailgate, lift upward until both sides latch into place.  CAUTION: After closin
Sample metadata     : {'chunk_id': '114_d0_c0', 'doc_id': '114_d0', 'row_id': '114', 'doc_pos': 0}


### Loading the and configuring the embedding model

In [36]:
import sys, os


def _add_ragbench_lib_to_path():
    for candidate in (os.getcwd(), os.path.join(os.getcwd(), "delucion_dataset")):
        if os.path.isdir(os.path.join(candidate, "ragbench_lib")) and candidate not in sys.path:
            sys.path.insert(0, candidate)
            return


_add_ragbench_lib_to_path()

from ragbench_lib.models import get_embedding_model

# Load OpenRouter API key
openrouter_api_key = os.environ.get("OPENROUTER_TOKEN")

# Available OpenRouter embedding models:
#   - "openai/text-embedding-3-small" (faster, cheaper)
#   - "openai/text-embedding-3-large" (better quality)
#   - "cohere/embed-english-v3.0" (good for retrieval)
#   - "mistral/mistral-embed" (lightweight)
#   - Check OpenRouter docs for other available models

embedding_model = get_embedding_model(openrouter_api_key)

print("✓ Embedding model loaded from OpenRouter")


✓ Embedding model loaded from OpenRouter


### Creating Vector DB with index and store the embeddings

In [ ]:
from ragbench_lib.vector_store import named_persist_dir, vector_store_names

_, collection_name = vector_store_names(DATASET_NAME, "base")
CHROMA_PATH = named_persist_dir(f"{DATASET_NAME}_base")

# Build index here, then download it from the output panel
vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name=collection_name,
    persist_directory=CHROMA_PATH,
)

### RAG Pipeline

In [38]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from ragbench_lib.models import get_generation_llm
from ragbench_lib.generation_prompt import RAG_GENERATION_PROMPT

llm = get_generation_llm(openrouter_token)

# ── Prompt ───────────────────────────────────────────────────────────────
prompt = RAG_GENERATION_PROMPT

# ── Retriever (replaces your retrieve() function) ───────────────────────────
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 20, "lambda_mult": 0.7},
)

# ── format_docs (replaces your build_context()) ─────────────────────────────
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# ── LCEL chain (replaces your generate() function) ──────────────────────────
rag_chain = (
    {
        "context" : retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [39]:
question = docs_df["question"].iloc[19]

answer = rag_chain.invoke(question)

print(f"Question     : {question}")
print(f"\nGenerated    :\n{answer}")
print(f"\nGround Truth :\n{docs_df['response'].iloc[19]}")

Question     : how to calculate the gross trailer weight?

Generated    :
According to the context, the Gross Trailer Weight (GTW) is calculated by adding the weight of the trailer, plus the weight of all cargo, consumables, and equipment (permanent or temporary) loaded in or on the trailer in its "loaded and ready for operation" condition.

Ground Truth :
To calculate the gross trailer weight (GTW), the recommended way is to put your fully loaded trailer on a vehicle scale where the entire weight of the trailer must be supported by the scale. The GTW is the weight of the trailer plus the weight of all cargo, consumables, and equipment (permanent or temporary) loaded in or on the trailer in its "loaded and ready for operation" condition. This measurement should be taken when the trailer is fully loaded and ready to be towed.


### Evaluators

### Evaluation: Llama 70B Annotation vs RAGBench Ground Truth

This section implements the RAGBench annotation prompt structure for Llama 70B and compares outputs against pre-computed RAGBench scores.

#### Step 1: Llama 70B Setup & Annotation Utilities

In [40]:
!pip install nltk


[notice] A new release of pip is available: 25.2 -> 26.2
[notice] To update, run: pip install --upgrade pip


In [41]:
openrouter_api_key = os.environ['OPENROUTER_TOKEN']

In [42]:
from ragbench_lib.models import get_judge_llm
from ragbench_lib.chunking import get_sentences
from ragbench_lib.trace_eval import format_documents_with_keys
import json, re

# Initialize Llama 70B for annotation
llama_judge = get_judge_llm(openrouter_api_key)

print("✓ Initialized Llama 70B annotation utilities")


✓ Initialized Llama 70B annotation utilities


#### Step 2: RAGBench Annotation Prompt (Section 7.4)

In [43]:
# The annotation prompt itself now lives in ragbench_lib.trace_eval.ANNOTATION_PROMPT_TEMPLATE
# (used internally by annotate_response_for_metrics below).
from ragbench_lib.trace_eval import ANNOTATION_PROMPT_TEMPLATE

print("✓ Loaded RAGBench annotation prompt (ragbench_lib.trace_eval)")


✓ Loaded RAGBench annotation prompt (ragbench_lib.trace_eval)


#### Step 3: Annotation Function with JSON Parsing

In [44]:
from ragbench_lib.trace_eval import annotate_response_for_metrics as _annotate_response_for_metrics


def annotate_response_for_metrics(documents: list[str], question: str, response: str) -> dict:
    """Annotate YOUR RAG response to extract relevant and utilized sentence keys.
    Returns structured data for TRACe metric calculation.
    """
    return _annotate_response_for_metrics(llama_judge, documents, question, response)


print("✓ Annotation function ready")


✓ Annotation function ready


In [45]:
# ── TRACe Metric Calculation Functions (Paper Formulas) ────────────────────
from ragbench_lib.trace_eval import (
    compute_context_relevance as compute_context_relevance_from_annotation,
    compute_utilization as compute_utilization_from_annotation,
    compute_completeness as compute_completeness_from_annotation,
    compute_adherence as compute_adherence_from_annotation,
)

print("✓ TRACe metric functions defined")


✓ TRACe metric functions defined


In [46]:
print("\n" + "="*80)
print("COMPUTING TRACE METRICS FOR RAG PIPELINE")
print("="*80 + "\n")

# Get unique questions (one per row_id, since questions are duplicated per document)
# Use up to 40 samples or however many are available
num_samples = 40
unique_samples = docs_df.drop_duplicates(subset=['row_id']).head(num_samples).reset_index(drop=True)
print(f"Testing on {len(unique_samples)} unique samples\n")

results = []

for i, row in unique_samples.iterrows():
    question = row['question']
    
    print(f"[{i+1}/{len(unique_samples)}] Q: {question[:70]}...")
    
    # ✅ STEP 1: RUN YOUR RAG PIPELINE
    try:
        my_response = rag_chain.invoke(question)
        print(f"  My response: {my_response[:70]}...")
    except Exception as e:
        print(f"  ✗ RAG failed: {str(e)[:60]}")
        continue
    
    # Retrieve relevant documents for annotation
    retrieved_docs = retriever.invoke(question)
    retrieved_texts = [doc.page_content for doc in retrieved_docs]
    
    # ✅ STEP 2: ANNOTATE YOUR RESPONSE TO GET RELEVANT & UTILIZED KEYS
    print(f"  Annotating response...", end=" ")
    annotation = annotate_response_for_metrics(retrieved_texts, question, my_response)
    
    if not annotation['success']:
        print(f"✗ Failed: {annotation.get('error', 'unknown')[:50]}")
        continue
    
    print("✓")
    
    # Show what the judge extracted
    print(f"    Relevant keys: {annotation['relevant_keys']}")
    print(f"    Utilized keys: {annotation['utilized_keys']}")
    
    # ✅ STEP 3: COMPUTE TRACE METRICS FROM ANNOTATION
    context_relevance = compute_context_relevance_from_annotation(retrieved_texts, annotation)
    utilization = compute_utilization_from_annotation(retrieved_texts, annotation)
    completeness = compute_completeness_from_annotation(annotation)
    adherence = compute_adherence_from_annotation(annotation)
    
    # Display computed metrics
    print(f"    Computed TRACe Metrics:")
    print(f"      Context Relevance: {context_relevance:.4f}")
    print(f"      Utilization:       {utilization:.4f}")
    print(f"      Completeness:      {completeness:.4f}")
    print(f"      Adherence:         {adherence:.4f}")
    print()
    
    # Store results
    results.append({
        'question_idx': i,
        'question': question,
        'my_response': my_response,
        'context_relevance': context_relevance,
        'utilization': utilization,
        'completeness': completeness,
        'adherence': adherence,
    })

print("✓ All responses processed\n")


COMPUTING TRACE METRICS FOR RAG PIPELINE

Testing on 40 unique samples

[1/40] Q: What if I fail to latch the tailgate properly?...
  My response: If you fail to latch the tailgate properly, it could result in damage ...
  Annotating response... ✓
    Relevant keys: ['0b', '1b', '2c']
    Utilized keys: ['0b']
    Computed TRACe Metrics:
      Context Relevance: 0.2500
      Utilization:       0.0833
      Completeness:      0.3333
      Adherence:         1.0000

[2/40] Q: What kind of safety features are implemented in this car?...
  My response: The safety features implemented in this car include:

1. Safety/Drivin...
  Annotating response... ✓
    Relevant keys: ['0a', '0b', '1a', '2a']
    Utilized keys: ['0a', '1a', '2a']
    Computed TRACe Metrics:
      Context Relevance: 0.8000
      Utilization:       0.6000
      Completeness:      0.7500
      Adherence:         1.0000

[3/40] Q: When will the Automatic SOS be triggered?...
  My response: The Automatic SOS will be triggere

#### Step 6: Summary & Next Steps

In [47]:
import numpy as np

print("\n" + "="*100)
print("TRACE METRICS SUMMARY")
print("="*100 + "\n")

# Build metrics table
metrics_rows = []

for result in results:
    metrics_rows.append({
        'Sample': result['question_idx'] + 1,
        'Question': result['question'][:40] + '...',
        'Context Relevance': f"{result['context_relevance']:.4f}",
        'Utilization': f"{result['utilization']:.4f}",
        'Completeness': f"{result['completeness']:.4f}",
        'Adherence': f"{result['adherence']:.4f}",
    })

# Display table
df_metrics = pd.DataFrame(metrics_rows)
display(df_metrics)

# Aggregate statistics
print("\n" + "="*100)
print("AVERAGE TRACE METRICS ACROSS ALL SAMPLES")
print("="*100 + "\n")

# Extract numeric values
rel_scores = [r['context_relevance'] for r in results]
util_scores = [r['utilization'] for r in results]
comp_scores = [r['completeness'] for r in results]
adh_scores = [r['adherence'] for r in results]

print("Average Scores (↑ higher = better):")
print(f"  Context Relevance:  {np.mean(rel_scores):.4f}  ± {np.std(rel_scores):.4f}")
print(f"  Utilization:        {np.mean(util_scores):.4f}  ± {np.std(util_scores):.4f}")
print(f"  Completeness:       {np.mean(comp_scores):.4f}  ± {np.std(comp_scores):.4f}")
print(f"  Adherence:          {np.mean(adh_scores):.4f}  ± {np.std(adh_scores):.4f}")

print("\n" + "-"*100)
print("Metric Interpretations:")
print("  Context Relevance: Fraction of retrieved documents that are relevant to the question")
print("  Utilization: Fraction of retrieved documents actually used in the response")
print("  Completeness: Coverage of relevant information (how many relevant docs were utilized)")
print("  Adherence: Whether the response is supported by the retrieved context (0 or 1)")
print("-"*100)


TRACE METRICS SUMMARY



,Sample,Question,Context Relevance,Utilization,Completeness,Adherence
0,1,What if I fail to latch the tailgate pro...,0.2500,0.0833,0.3333,1.0000
1,2,What kind of safety features are impleme...,0.8000,0.6000,0.7500,1.0000
2,3,When will the Automatic SOS be triggered...,0.1429,0.1429,1.0000,1.0000
3,4,What happens if I accidentally push the ...,0.3333,0.1111,0.3333,1.0000
4,5,What is the DEF?...,0.0000,0.0000,0.0000,0.0000
5,6,What may cause erratic or noisy performa...,0.3333,0.3333,1.0000,1.0000
6,7,how to calculate the gross trailer weigh...,0.1250,0.1250,1.0000,1.0000
7,8,What can the ASIST button do?...,0.5000,0.5000,1.0000,1.0000
8,9,What does the Door Off Mirror Kit do?...,0.6000,0.4000,0.6667,1.0000
9,10,Can I manually activate or deactivate th...,0.1818,0.0000,0.0000,1.0000



AVERAGE TRACE METRICS ACROSS ALL SAMPLES

Average Scores (↑ higher = better):
  Context Relevance:  0.3637  ± 0.2125
  Utilization:        0.2196  ± 0.1520
  Completeness:       0.6169  ± 0.3341
  Adherence:          0.8462  ± 0.3608

----------------------------------------------------------------------------------------------------
Metric Interpretations:
  Context Relevance: Fraction of retrieved documents that are relevant to the question
  Utilization: Fraction of retrieved documents actually used in the response
  Completeness: Coverage of relevant information (how many relevant docs were utilized)
  Adherence: Whether the response is supported by the retrieved context (0 or 1)
----------------------------------------------------------------------------------------------------


In [49]:
run_quick_experiment('M3', docs_df, embedding_model, llm, prompt, 10)



QUICK EXPERIMENT: M3 (10 samples)


REBUILDING VECTOR STORE WITH: M3

Applying M3 chunking strategy... ✓ Created 194 chunks
Deleting old chromadb... ✓
Building new vector store with 194 documents... 

InternalError: Database error: error returned from database: (code: 1032) attempt to write a readonly database